# QUT-DV25 preview notebook

This notebook downloads the **QUT-DV25** public repository, discovers tabular dataset files inside it, loads the most promising processed table automatically, and visualizes the first examples.

This is the public dataset artifact aligned with the DySec/QUT dynamic-analysis line of work.

**Source**  
- GitHub repository: `tanzirmehedi/QUT-DV25`  
- Public repo page states the dataset contains **14,271 observations**, **36 variables**, and includes **raw trace files and processed CSV data**. citeturn791770view0turn483053view4

In [ ]:
!pip -q install pandas pyarrow requests

In [ ]:
import json
import ast
from pathlib import Path
from typing import Any
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 200)

def truncate(text: Any, n: int = 300) -> str:
    if text is None:
        return ""
    s = str(text)
    return s if len(s) <= n else s[:n] + " …"

def maybe_parse_json(x: Any):
    if isinstance(x, (dict, list)):
        return x
    if not isinstance(x, str):
        return x
    x = x.strip()
    if not x:
        return x
    for fn in (json.loads, ast.literal_eval):
        try:
            return fn(x)
        except Exception:
            pass
    return x

def first_dialogue_excerpt(obj: Any, max_items: int = 4) -> str:
    parsed = maybe_parse_json(obj)
    if isinstance(parsed, list):
        parts = []
        for item in parsed[:max_items]:
            if isinstance(item, dict):
                role = item.get("role") or item.get("userType") or item.get("type") or "item"
                content = item.get("content") or item.get("text") or item.get("message") or item.get("action") or item.get("thought") or item
                parts.append(f"{role}: {truncate(content, 180)}")
            else:
                parts.append(truncate(item, 180))
        return "\n".join(parts)
    if isinstance(parsed, dict):
        return truncate(json.dumps(parsed, indent=2), 500)
    return truncate(parsed, 500)

def choose_first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None


import os
import requests
import zipfile
import tarfile
import io
from pathlib import Path

def download_file(url: str, dest: str, chunk_size: int = 2**20):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        print(f"Using cached file: {dest}")
        return dest
    print(f"Downloading {url} -> {dest}")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    return dest

def unzip_file(zip_path: str, out_dir: str):
    zip_path = Path(zip_path)
    out_dir = Path(out_dir)
    if out_dir.exists() and any(out_dir.iterdir()):
        print(f"Using cached extraction: {out_dir}")
        return out_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(out_dir)
    return out_dir

In [ ]:
REPO_ZIP_URL = "https://github.com/tanzirmehedi/QUT-DV25/archive/refs/heads/main.zip"
CACHE_DIR = Path("data_cache/qut_dv25")
ZIP_PATH = CACHE_DIR / "qut_dv25_main.zip"
EXTRACT_DIR = CACHE_DIR / "repo"

download_file(REPO_ZIP_URL, ZIP_PATH)
unzip_file(ZIP_PATH, EXTRACT_DIR)

roots = [p for p in EXTRACT_DIR.iterdir() if p.is_dir()]
root = roots[0] if len(roots) == 1 else EXTRACT_DIR
print("Repository root:", root)

In [ ]:
candidate_files = []
for suffix in (".csv", ".tsv", ".json", ".jsonl", ".parquet"):
    candidate_files.extend(root.rglob(f"*{suffix}"))

candidate_files = sorted(candidate_files, key=lambda p: (p.suffix, p.stat().st_size), reverse=True)

preview = pd.DataFrame(
    {
        "path": [str(p.relative_to(root)) for p in candidate_files[:50]],
        "suffix": [p.suffix for p in candidate_files[:50]],
        "size_mb": [round(p.stat().st_size / 1e6, 3) for p in candidate_files[:50]],
    }
)
display(preview.head(20))
print(f"Found {len(candidate_files)} candidate data files.")

In [ ]:
def score_candidate(path: Path) -> tuple:
    name = str(path).lower()
    score = 0
    for token, val in [
        ("processed", 5), ("final", 5), ("dataset", 4), ("label", 4),
        ("trace", 3), ("feature", 3), ("report", 2), ("analysis", -2),
        ("readme", -5), ("license", -5)
    ]:
        if token in name:
            score += val
    if path.suffix == ".parquet":
        score += 4
    if path.suffix == ".csv":
        score += 3
    score += min(path.stat().st_size / 1e6, 500) / 20
    return (score, path.stat().st_size)

ranked = sorted(candidate_files, key=score_candidate, reverse=True)
ranked_df = pd.DataFrame({
    "path": [str(p.relative_to(root)) for p in ranked[:20]],
    "score": [score_candidate(p)[0] for p in ranked[:20]],
    "size_mb": [round(p.stat().st_size / 1e6, 3) for p in ranked[:20]],
})
display(ranked_df)
best_path = ranked[0]
print("Selected file:", best_path)

In [ ]:
def load_table(path: Path) -> pd.DataFrame:
    if path.suffix == ".csv":
        return pd.read_csv(path)
    if path.suffix == ".tsv":
        return pd.read_csv(path, sep="\t")
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".json":
        try:
            return pd.read_json(path)
        except ValueError:
            with open(path, "r", encoding="utf-8") as f:
                obj = json.load(f)
            if isinstance(obj, dict):
                if all(isinstance(v, dict) for v in obj.values()):
                    return pd.DataFrame.from_dict(obj, orient="index").reset_index(names="key")
                return pd.json_normalize(obj)
            if isinstance(obj, list):
                return pd.json_normalize(obj)
            return pd.DataFrame({"value": [obj]})
    if path.suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    raise ValueError(f"Unsupported suffix: {path.suffix}")

df = load_table(best_path)
print(df.shape)
display(df.head())

In [ ]:
summary = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(t) for t in df.dtypes],
    "n_unique": [df[c].nunique(dropna=True) if df[c].dtype == 'object' else None for c in df.columns],
    "n_missing": [int(df[c].isna().sum()) for c in df.columns],
})
display(summary.head(30))

In [ ]:
label_col = choose_first_existing(df.columns, [
    "label", "Label", "class", "Class", "malicious", "is_malicious", "category", "Category"
])
if label_col:
    display(df[label_col].value_counts(dropna=False).rename_axis(label_col).reset_index(name="count").head(20))
else:
    print("No obvious label column found.")

display(df.head(10))

If the automatically selected file is not the one you want, re-run the load cell with another path from the ranked table above.